Gold Aggregation Layer Coded in Pyspark

In [0]:
df_customers = spark.read.parquet("s3a://wealth-de-poc-astitva/silver/customers/")
df_txn = spark.read.parquet("s3a://wealth-de-poc-astitva/silver//")
df_stock = spark.read.parquet("s3a://wealth-de-poc-astitva/silver/stock_api/")

df_txn.show(5)

+-----------+------+------+--------+-----+--------------------+--------+----------+-------+---+------------+------------------+
|customer_id|txn_id|ticker|quantity|price|           timestamp|txn_type| load_date|   name|age|risk_profile|customer_load_date|
+-----------+------+------+--------+-----+--------------------+--------+----------+-------+---+------------+------------------+
|       C506|    T5|  INTC|       5|  300|2026-04-24 07:01:...|     BUY|2026-04-24|  Alice| 25|     unknown|        2026-04-20|
|        C40|   T31|  AMZN|       5|  100|2026-04-23 21:14:...|    SELL|2026-04-24|  Alice| 25|      medium|        2026-04-20|
|       C338|   T32|  INTC|      10|  100|2026-04-24 01:32:...|    SELL|2026-04-24|Charlie| 25|     unknown|        2026-04-20|
|       C827|  T132|  AAPL|      10|  100|2026-04-23 23:39:...|    SELL|2026-04-24|    Bob| 25|     unknown|        2026-04-20|
|       C481|  T134|  INTC|       1|  100|2026-04-23 21:42:...|     BUY|2026-04-24|Charlie| 25|         

In [0]:
from pyspark.sql.functions import col, when

df_txn_norm = df_txn.withColumn(
    "signed_quantity",
    when(col("txn_type") == 'SELL',-col("quantity"))
    .otherwise(col('quantity'))
)

df_txn_norm.show()

+-----------+------+------+--------+-----+--------------------+--------+----------+-------+---+------------+------------------+---------------+
|customer_id|txn_id|ticker|quantity|price|           timestamp|txn_type| load_date|   name|age|risk_profile|customer_load_date|signed_quantity|
+-----------+------+------+--------+-----+--------------------+--------+----------+-------+---+------------+------------------+---------------+
|       C506|    T5|  INTC|       5|  300|2026-04-24 07:01:...|     BUY|2026-04-24|  Alice| 25|     unknown|        2026-04-20|              5|
|        C40|   T31|  AMZN|       5|  100|2026-04-23 21:14:...|    SELL|2026-04-24|  Alice| 25|      medium|        2026-04-20|             -5|
|       C338|   T32|  INTC|      10|  100|2026-04-24 01:32:...|    SELL|2026-04-24|Charlie| 25|     unknown|        2026-04-20|            -10|
|       C827|  T132|  AAPL|      10|  100|2026-04-23 23:39:...|    SELL|2026-04-24|    Bob| 25|     unknown|        2026-04-20|         

Net Holdings After Buys/Sells

In [0]:
from pyspark.sql.functions import sum

df_positions = df_txn_norm.groupBy("customer_id","ticker").agg(
    sum("signed_quantity").alias("net_quantity")
)
df_positions.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- net_quantity: long (nullable = true)



In [0]:
df_positions = df_positions.filter(col('net_quantity')>0)
df_positions.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- net_quantity: long (nullable = true)



In [0]:
from pyspark.sql.functions import sum as _sum
from pyspark.sql.functions import col


df_buys = df_txn.filter(col('txn_type')=='BUY')

df_avg_price = df_buys.groupBy("customer_id","ticker").agg(
    (_sum(col('price')*col('quantity'))/ _sum("quantity")).alias("avg_buy_price"))

df_avg_price.show(5)

+-----------+------+-------------+
|customer_id|ticker|avg_buy_price|
+-----------+------+-------------+
|       C120|  AAPL|        200.0|
|        C17|  MSFT|        200.0|
|       C340|  TSLA|        300.0|
|       C849|   AMD|        200.0|
|        C14|   AMD|        300.0|
+-----------+------+-------------+
only showing top 5 rows


Combining Customer Positions and Cost Basis

In [0]:
df_avg_price = df_avg_price.select(
    "customer_id",
    "ticker",
    "avg_buy_price"
)
df_positions = df_positions.join(
    df_avg_price,
    on=["customer_id","ticker"],
    how="inner"
)

#df_positions.show()
df_positions.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- ticker: string (nullable = true)
 |-- net_quantity: long (nullable = true)
 |-- avg_buy_price: double (nullable = true)



Get Latest Market Prices


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col

window_spec = Window.partitionBy("ticker").orderBy(col("last_trade_time").desc())

df_latest_prices = df_stock.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank")==1
).drop("rank")

In [0]:
df_latest_prices.show(5)

+------+------+-------+-------------------+--------+--------+-------+----------+----------+--------------------+
|ticker| price| volume|    last_trade_time|currency|day_high|day_low|day_change|      date| ingestion_timestamp|
+------+------+-------+-------------------+--------+--------+-------+----------+----------+--------------------+
|  AAPL|273.06|1227192|2026-04-20 15:59:59|     USD|  274.28| 270.33|      1.05|2026-04-23|2026-04-23 14:22:...|
|   AMD|275.02| 441638|2026-04-20 15:59:59|     USD|  287.52| 272.19|     -1.23|2026-04-23|2026-04-23 14:22:...|
|  AMZN|248.34|1451531|2026-04-20 15:59:59|     USD|  250.08| 245.37|     -0.87|2026-04-23|2026-04-23 14:22:...|
| GOOGL|337.35| 565160|2026-04-20 16:00:00|     USD|  341.26| 336.62|     -1.28|2026-04-23|2026-04-23 14:22:...|
|  INTC| 65.73|3252033|2026-04-20 16:00:00|     USD|   69.15|   65.1|     -4.21|2026-04-23|2026-04-23 14:22:...|
+------+------+-------+-------------------+--------+--------+-------+----------+----------+-----

In [0]:


df_latest_prices = df_latest_prices.select(
    "ticker",
    col("price").alias("current_price")
)

df_positions = df_positions.join(
    df_latest_prices,
    on='ticker',
    how="inner"
)
df_positions = df_positions.withColumnRenamed("price", "current_price")

df_positions.show(4)
df_positions.printSchema()

+------+-----------+------------+-------------+-------------+
|ticker|customer_id|net_quantity|avg_buy_price|current_price|
+------+-----------+------------+-------------+-------------+
|  AAPL|       C120|           1|        200.0|       273.06|
|  MSFT|        C17|           5|        200.0|       418.09|
|  TSLA|       C340|           4|        300.0|       392.42|
|   AMD|       C849|           5|        200.0|       275.02|
+------+-----------+------------+-------------+-------------+
only showing top 4 rows
root
 |-- ticker: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- net_quantity: long (nullable = true)
 |-- avg_buy_price: double (nullable = true)
 |-- current_price: double (nullable = true)



In [0]:
from pyspark.sql.functions import col

df_positions = df_positions.withColumn(
    "portfolio_value",
    col("net_quantity") * col("current_price")
)

df_positions = df_positions.withColumn(
    "PnL",
    (col("current_price") - col("avg_buy_price")) * col("net_quantity")
)


df_positions.show()

+------+-----------+------------+-------------+-------------+------------------+-------------------+
|ticker|customer_id|net_quantity|avg_buy_price|current_price|   portfolio_value|                PnL|
+------+-----------+------------+-------------+-------------+------------------+-------------------+
|  AAPL|       C120|           1|        200.0|       273.06|            273.06|              73.06|
|  MSFT|        C17|           5|        200.0|       418.09|           2090.45| 1090.4499999999998|
|  TSLA|       C340|           4|        300.0|       392.42|           1569.68| 369.68000000000006|
|   AMD|       C849|           5|        200.0|       275.02|            1375.1|  375.0999999999999|
|   AMD|        C14|          10|        300.0|       275.02|            2750.2|-249.80000000000018|
|  AMZN|       C162|           5|        100.0|       248.34|            1241.7|              741.7|
|  NVDA|       C739|          10|        300.0|       202.12|            2021.2|           

Aggregating Per Customer

In [0]:
df_gold = df_positions.groupBy("customer_id").agg(
    _sum("portfolio_value").alias("total_portfolio_value"),
    _sum("PnL").alias("total_pnl")
)

df_gold.show(5)

+-----------+---------------------+------------------+
|customer_id|total_portfolio_value|         total_pnl|
+-----------+---------------------+------------------+
|       C609|   3452.1400000000003|2352.1400000000003|
|       C803|              8292.55|1792.5499999999997|
|       C907|             10759.62| 959.6199999999994|
|       C677|              4518.25|3218.2499999999995|
|       C204|             13007.07|           4807.07|
+-----------+---------------------+------------------+
only showing top 5 rows


In [0]:

df_customers_sel = df_customers.select(
    "customer_id",
    "name",
    "risk_profile"
)
df_gold = df_gold.join(
    df_customers_sel,
    on="customer_id",
    how="left"
)

In [0]:
df_gold.show(4)

+-----------+---------------------+------------------+-------+------------+
|customer_id|total_portfolio_value|         total_pnl|   name|risk_profile|
+-----------+---------------------+------------------+-------+------------+
|       C609|   3452.1400000000003|2352.1400000000003|    Bob|      medium|
|       C803|              8292.55|1792.5499999999997|Charlie|      medium|
|       C907|             10759.62| 959.6199999999994|    Bob|        high|
|       C677|              4518.25|3218.2499999999995|Charlie|         low|
+-----------+---------------------+------------------+-------+------------+
only showing top 4 rows


In [0]:
df_gold.write \
    .mode("overwrite") \
    .parquet("s3a://wealth-de-poc-astitva/gold/customer_portfolio/")